In [ ]:
# manually defined Parameters+++==============
# FISH AND RECORDING
REC_NAME = '2026-02-25_mb_fish1_rec2'
# REC_NAME = '2026-03-04_mb_fish8_rec2'

# CALCIUM DATA
PLANE = 0
IMG_RATE = 1.9988  # fish1_rec2
# IMG_RATE = 1.9957 # fish8_rec2
LOAD_DFF = True  # if dff has been calculated before

# EYETRACKING DATA
EYEPOS_QUADRANTS = (.1, .1, -.1, -.1)

In [1]:
from preprocess import *

import pickle
from os.path import join
from scipy.interpolate import interp1d

In [ ]:
# LOAD DATA
rec_path = join('data', REC_NAME)
save_path = join('results', REC_NAME, 'plane' + str(PLANE))
camera = h5py.File(join(rec_path, 'Camera.hdf5'), 'r')
fluorescence, rec, phase, _ = digest_folder(rec_path, IMG_RATE, PLANE)

In [ ]:
# PREPROCESSING CMN STIMULI
# add resampled CMN data to resampled time domain for each phase
# (info: time domain is resampled from ~2.1Hz to 10Hz in digest_folder())
process_recording(rec, phase, radial_bin_num=16)

In [ ]:
# PREPROCESSING FLUORESCENCE TRACE
if LOAD_DFF:
    dff_original = np.load(join(save_path, "dff_original.npy"))
    dff_resampled = (interp1d(
        rec['ca_times'],
        dff_original,
        kind='nearest')(rec['time_resampled']))
else:
    dff_original, dff_resampled = calc_dff(
        rec,
        fluorescence,
        rec['imaging_rate'])
    Path(save_path).mkdir(parents=True, exist_ok=True)
    np.save(join(save_path, "dff_original.npy"), dff_original)

In [ ]:
# EYE POSITION SELECTION
q1_mask, q3_mask = calc_eyepos_masks(
    camera['eyepos_ang_le_pos_0'],
    camera['eyepos_ang_re_pos_0'],
    camera['fish_embedded_frame_time'],
    rec['time_resampled'],
    q1_min_left = EYEPOS_QUADRANTS[0],
    q1_min_right = EYEPOS_QUADRANTS[1],
    q3_max_left = EYEPOS_QUADRANTS[2],
    q3_max_right = EYEPOS_QUADRANTS[3],
    verbose=True)

In [ ]:
# SELECT NEURON
i = 0

In [ ]:
# calcium events
rec['signal_selection'], rec['signal_length'], rec['signal_proportion'], rec['signal_dff_mean'] \
    = detect_events(rec['cmn_phase_selection'], dff_resampled[i], rec['sample_rate'])

# subselect neurons

## 8% threshold for neural activity

In [ ]:
proportions=[]
for dff_i in dff_resampled:
    signal_selection, signal_length, signal_proportion, signal_dff_mean\
        =detect_events(
        rec['cmn_phase_selection'],
        dff_i,
        rec['sample_rate'])
    proportions.append(signal_proportion)

In [ ]:
prop=np.array(proportions)

In [ ]:
plt.hist(prop, bins=80)

In [ ]:
phase_visual = [[name, phase[name]['__visual_name']] for name in phase.keys() if name.startswith('phase')]
phase_visual = np.array(phase_visual)
phase_visual

## based on response intensitites > 3

In [ ]:
phase_id=99

In [ ]:
dff_original[:,phase['in_phase_idcs_'+str(phase_id)]].max(axis=1).shape

In [ ]:
(dff_original[:,phase['in_phase_idcs_'+str(phase_id)]].max(axis=1)-
 dff_original[:,phase['in_phase_idcs_'+str(phase_id)][0]]).shape

In [ ]:
R=np.stack([np.maximum(0,
                       dff_original[:,phase['in_phase_idcs_'+str(phase_id)]].max(axis=1)-
                       dff_original[:,phase['in_phase_idcs_'+str(phase_id)][0]]
                       )
            for phase_id in range(105)], axis=1)
R.shape

In [ ]:
strong_response_indices_mask = R.max(axis=0)>3
strong_response_indices_mask.sum()

In [ ]:
plt.hist(R.flatten(), bins=2000)

## exlude with drift or whitening from first to last 16 seconds

In [ ]:
t = rec['time_resampled']
t0 = t[0]

In [ ]:
# masks for fist and last 16 seconds
start, end= (t-t[0] < 16), (t > t[-1] - 16.)

In [ ]:
mu_end, mu_start=dff_resampled[:,end].mean(axis=1), dff_resampled[:,start].mean(axis=1)
std_end, std_start=dff_resampled[:,end].std(axis=1), dff_resampled[:,start].std(axis=1)

In [ ]:
np.sum(np.abs(mu_start-mu_end)>4*mu_start), sum((std_end/std_start) > 4)